<a href="https://colab.research.google.com/github/cook1e-0707/practicalAI-lab/blob/main/day3/student/week1_day3_yourname.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

# Week 1 · Day 3 Student Lab
## Unsupervised Learning with K-Means and Recursive K-Means

**Before you start:** Choose **File → Save a copy in Drive**.
Rename it `week1_day3_yourname.ipynb`.


## Today

Day 2 models learned from inputs and known target answers.
Day 3 removes the target answers and asks:

> Can an algorithm find useful groups from input patterns alone?

We will use:

1. **K-Means:** directly create K clusters;
2. **Recursive/Bisecting K-Means:** repeatedly divide existing
   clusters into two smaller clusters.

Our datasets include coordinates, handwritten images,
spoken-digit recordings, curved shapes, hierarchical points,
and image pixels.


# Python Review: Tuple

A **tuple** stores an ordered group of values.

```python
point = (2.5, 4.0)
```

- A list uses square brackets: `[2.5, 4.0]`.
- A tuple normally uses parentheses: `(2.5, 4.0)`.
- A list can be changed after it is created.
- A tuple cannot be changed after it is created.
- Tuple positions start at `0`, just like list positions.

The commas separate the tuple values. Here, position `0` is the
x coordinate and position `1` is the y coordinate.

The `len(...)` function from Day 1 also works with a tuple. It
returns the number of values in that tuple.

Run the short review cell and focus on the tuple's parentheses,
fixed values, and position order.


In [ ]:
sample_point = (2.5, 4.0)

print("Tuple:", sample_point)
print("Python type:", type(sample_point))
print("Value at position 0:", sample_point[0])
print("Number of tuple values:", len(sample_point))


In the first K-Means case, each fixed center will be one
coordinate tuple such as `(-5, -2)`. Several center tuples can
also be stored together inside a larger tuple.


# Where Day 3 Data Comes From

In Day 1, Pandas read rows from a CSV file. Day 3 uses another
common method: Python asks scikit-learn to provide data directly.
The result is usually a NumPy array or a scikit-learn `Bunch`
object rather than a CSV file.

A `Bunch` is a container that keeps related dataset pieces
together. We access a piece with a dot, such as `digits.data`
for model rows or `digits.images` for image-shaped rows.

The [`sklearn.datasets`](https://scikit-learn.org/stable/api/sklearn.datasets.html)
package provides several kinds of dataset functions:

| Function name | What it does | Internet needed? |
|---|---|---|
| `make_*` | Generates artificial data when the cell runs | No |
| `load_*` | Loads a small dataset or sample bundled with scikit-learn | No |
| `fetch_*` | Downloads and caches a larger dataset | Usually yes on the first run |

**Cache** means saving a downloaded copy in the current Python
environment so it can be reused. A local environment may keep
that copy; Colab files disappear when its temporary runtime is
reset.

Examples of the larger `fetch_*` collection include forest
measurements, news articles, face images, species locations,
and California housing data. Some contain hundreds of thousands
of rows. They are useful in longer projects, but downloads and
longer processing times are not needed for today's three-hour
lab. See the
[scikit-learn real-world dataset guide](https://scikit-learn.org/stable/datasets/real_world.html).

**Important:** generated data is not real-world evidence. It is
designed so that one idea can be seen clearly. For every case,
we will identify whether the data is generated, bundled, or
derived from real observations.


# Unsupervised Learning

In supervised learning:

```text
X inputs + y correct answers → model
```

In today's unsupervised clustering:

```text
X inputs only → cluster assignments and cluster centers
```

There is no target column telling the model the correct group.
A cluster is a group created from a similarity rule.

Cluster numbers are arbitrary identifiers. Cluster `0` is not
better, smaller, or more correct than cluster `1`.


# K-Means: Complete Coordinate Example

## Dataset description

`make_blobs(...)` generates 450 artificial coordinate rows
inside Colab. Each row contains an `x` position and a `y`
position. The dataset is designed for teaching and is not a
real sensor dataset.



## Input

- 450 observations;
- two numeric features per observation: `x` and `y` coordinates;
- array shape: `(450, 2)`;
- no target answers are passed to K-Means.

## Output

- one cluster number for every observation: shape `(450,)`;
- three learned centers: shape `(3, 2)`;
- inertia and the number of update rounds.

`make_blobs(...)` creates numeric coordinate data for this
teaching example. `KMeans(...)` creates the model that will
group similar rows.
We import both functions in the next cell. The model itself is
created after we first look at the input points.

We import each package when it is first used.

### Build the coordinate dataset

- `make_blobs(...)` returns generated input rows and generated
  group numbers;
- `n_samples=450` requests 450 rows;
- `centers=coordinate_data_centers` places the generated groups
  around the three coordinate tuples;
- `cluster_std=[...]` controls each group's spread;
- `random_state=42` repeats the same generated rows.

The generated group numbers are stored in
`unused_hidden_groups`; K-Means will not receive them.


In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

print("Imported package:", np.__name__)
print("NumPy version:", np.__version__)
print("Imported package:", matplotlib.__name__)
print("Matplotlib version:", matplotlib.__version__)
print("Imported package:", sklearn.__name__)
print("scikit-learn version:", sklearn.__version__)

coordinate_data_centers = (
    (-5, -2),
    (0, 5),
    (5, -1),
)

coordinate_points, unused_hidden_groups = make_blobs(
    n_samples=450,
    centers=coordinate_data_centers,
    cluster_std=[1.0, 1.2, 0.9],
    random_state=42,
)

print("Tuple of data centers:", coordinate_data_centers)
print("Input shape:", coordinate_points.shape)
print("First three input rows:")
print(coordinate_points[:3])


The next plot shows only the input. It does not use hidden group
answers or model output.

The plotting code is provided. Focus on the two input columns:
`[:, 0]` supplies x and `[:, 1]` supplies y. Every point uses
one color because no cluster labels exist yet.


In [ ]:
plt.scatter(
    coordinate_points[:, 0],
    coordinate_points[:, 1],
    color="#78909C",
    alpha=0.7,
)
plt.xlabel("Feature 1: x coordinate")
plt.ylabel("Feature 2: y coordinate")
plt.title("Input before clustering")
plt.show()


## How K-Means Works

K-Means repeats four steps:

1. choose K starting centers;
2. assign every point to its nearest center;
3. replace each center with the mean of its assigned points;
4. repeat assignment and update until centers stop changing or
   `max_iter` is reached.

Distance uses all input features. K-Means works most naturally
when clusters are compact and roughly round.

### Create and train the model

- `KMeans(...)` creates the model;
- `n_clusters=3` requests three groups;
- `init="k-means++"` chooses spread-out starting centers;
- `n_init=10` tries ten initial arrangements;
- `max_iter=300` limits update rounds;
- `random_state=42` makes the starting choices repeatable;
- `.fit(X)` learns the centers from the input rows;
- `.predict(X)` assigns rows to the nearest learned center;
- `.fit_predict(X)` is a shortcut that fits and immediately
  returns cluster numbers for the same rows;
- `.cluster_centers_` stores the learned centers;
- `.inertia_` stores the total squared distance from rows to
  their assigned centers;
- `.n_iter_` stores the number of completed update rounds;
- `np.bincount(labels)` counts how many rows received each
  non-negative cluster number;
- `round(number, 2)` returns that number rounded to two decimal
  places for easier reading. It changes only the displayed
  number here; it does not change the model's stored
  `.inertia_`.

We deliberately use `.fit(...)` followed by `.predict(...)`
here so both steps are visible. Later tasks may use the shorter
`.fit_predict(...)` form.


In [ ]:
coordinate_model = KMeans(
    n_clusters=3,
    init="k-means++",
    n_init=10,
    max_iter=300,
    random_state=42,
)
coordinate_model.fit(coordinate_points)
coordinate_labels = coordinate_model.predict(coordinate_points)
coordinate_centers = coordinate_model.cluster_centers_

print("Cluster-label shape:", coordinate_labels.shape)
print("Center shape:", coordinate_centers.shape)
print("Cluster sizes:", np.bincount(coordinate_labels))
# Show inertia with two decimal places; the model keeps
# its original full-precision inertia_ value.
print("Inertia:", round(coordinate_model.inertia_, 2))
print("Update rounds:", coordinate_model.n_iter_)


### View the clustering result

The plotting code is provided. Its important inputs are
`coordinate_labels`, which color the rows, and
`coordinate_centers`, which place the black X markers.


In [ ]:
plt.scatter(
    coordinate_points[:, 0],
    coordinate_points[:, 1],
    c=coordinate_labels,
    cmap="viridis",
    alpha=0.75,
)
plt.scatter(
    coordinate_centers[:, 0],
    coordinate_centers[:, 1],
    color="black",
    marker="X",
    s=260,
    label="Learned centers",
)
plt.xlabel("Feature 1: x coordinate")
plt.ylabel("Feature 2: y coordinate")
plt.title("K-Means output: labels and centers")
plt.legend()
plt.show()


## See the Internal Center Changes

The following provided inspection code starts from three fixed
centers and shows the result after one, two, and up to ten
update rounds.

`init=initial_centers` supplies the same starting positions.
`n_init=1` uses that initialization once. This is for inspecting
the algorithm; the main model above uses `k-means++`.

The plotting and list-management lines are provided. Follow the
model changes: every inspection model receives the same
`initial_centers`, while `max_iter` changes from 1 to 2 to 10.
`.labels_` and `.cluster_centers_` show the fitted result at
each limit.


In [ ]:
initial_centers = np.array([
    [-6.0, -4.0],
    [0.0, 7.0],
    [6.0, -3.0],
])
iteration_limits = [1, 2, 10]
inspection_models = []

for iteration_limit in iteration_limits:
    inspection_model = KMeans(
        n_clusters=3,
        init=initial_centers,
        n_init=1,
        max_iter=iteration_limit,
        random_state=42,
    )
    inspection_model.fit(coordinate_points)
    inspection_models.append(inspection_model)

figure, axes = plt.subplots(1, 4, figsize=(17, 4))

axes[0].scatter(
    coordinate_points[:, 0],
    coordinate_points[:, 1],
    color="#CFD8DC",
    s=18,
)
axes[0].scatter(
    initial_centers[:, 0],
    initial_centers[:, 1],
    color="red",
    marker="X",
    s=220,
)
axes[0].set_title("Starting centers")

for axis, model, limit in zip(
    axes[1:],
    inspection_models,
    iteration_limits,
):
    axis.scatter(
        coordinate_points[:, 0],
        coordinate_points[:, 1],
        c=model.labels_,
        cmap="viridis",
        s=18,
        alpha=0.65,
    )
    axis.scatter(
        model.cluster_centers_[:, 0],
        model.cluster_centers_[:, 1],
        color="red",
        marker="X",
        s=220,
    )
    axis.set_title(f"Up to {limit} update round(s)")

for axis in axes:
    axis.set_xlabel("x")
    axis.set_ylabel("y")

plt.tight_layout()
plt.show()


# Case 2 — Choose K with an Elbow Plot

## Dataset description

This case reuses the same 450 artificial coordinate rows from
the complete K-Means example. It does not load or generate
another dataset.



## Input

- the same `(450, 2)` coordinate array;
- candidate values `K = 2, 3, ..., 8`.

## Output

- one inertia value for every candidate K;
- an elbow plot;
- no target answers.

### Try several values of K

The model methods come directly from the complete coordinate
example:

- `n_clusters=candidate_k` uses the current loop value;
- `.fit(coordinate_points)` learns centers for that candidate;
- `.inertia_` reads the fitted model's total squared distance.

The loop and plotting lines are provided. Their role is to
repeat the same model process for K from 2 through 8 and display
the seven inertia values.

### Your coding task

Complete the intermediate model parameter, `.fit(...)`, and
stored inertia value.


In [ ]:
candidate_k_values = list(range(2, 9))
inertia_values = []

for candidate_k in candidate_k_values:
    candidate_model = KMeans(
        n_clusters=None,  # TODO: use candidate_k
        n_init=10,
        random_state=42,
    )
    # TODO: fit candidate_model with coordinate_points
    inertia_values.append(
        None  # TODO: store candidate_model.inertia_
    )

plt.plot(
    candidate_k_values,
    inertia_values,
    marker="o",
)
plt.xticks(candidate_k_values)
plt.xlabel("Number of clusters: K")
plt.ylabel("Inertia")
plt.title("Elbow plot")
plt.show()


<details>
<summary>Hint 1</summary>

During each loop, use `candidate_k` as the model's cluster count.
Fit that model with the same coordinate input used in the complete
example, then read the fitted model's inertia.
</details>

<details>
<summary>Hint 2: code shape</summary>

```python
candidate_model = KMeans(
    n_clusters=current_k,
    n_init=10,
    random_state=42,
)
candidate_model.fit(input_rows)
saved_values.append(candidate_model.inertia_)
```

Replace the descriptive names with this task's variable names.
</details>


**Interpretation:** Inertia always tends to decrease as K grows.
Look for a bend where adding more clusters produces much smaller
improvements. Here the main bend is near `K=3`.


# Case 3 — Cluster Handwritten Digit Images

## Dataset description

This is derived from **real human handwriting**. The
scikit-learn `load_digits()` version contains 1,797 images and
is a copy of the test portion of the UCI Optical Recognition of
Handwritten Digits dataset. The larger UCI dataset contains
5,620 digit records from 43 people. Its original `32 × 32`
bitmaps were summarized into `8 × 8` grids whose values range
from 0 through 16.

Sources:
[scikit-learn digits documentation](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html)
and
[UCI dataset description](https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits).



## Input

- 1,797 handwritten digit images;
- each image is `8 × 8` pixels;
- each model row contains 64 pixel values;
- model-input shape: `(1797, 64)`.

## Output

- one cluster number per image: shape `(1797,)`;
- ten center rows: shape `(10, 64)`;
- each center can be reshaped into an `8 × 8` average image.

The dataset contains target digits for research, but this lab
does not pass those targets to K-Means.

`load_digits()` provides the image dataset. It returns an
object containing both the flattened rows in `.data` and the
`8 × 8` images in `.images`.

### Load and display the digit images

- `load_digits()` returns the bundled `Bunch`;
- `digits.data` selects the 64-number model rows;
- `digits.images` selects the same data as `8 × 8` images.

The provided plotting loop displays ten example images. The
important pipeline output is `digit_values`, the `(1797, 64)`
table that will enter K-Means.


In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
digit_values = digits.data

print("Image-array shape:", digits.images.shape)
print("K-Means input shape:", digit_values.shape)

figure, axes = plt.subplots(2, 5, figsize=(10, 4))
for axis, image in zip(axes.ravel(), digits.images[:10]):
    axis.imshow(image, cmap="gray")
    axis.axis("off")
figure.suptitle("Ten input images")
plt.tight_layout()
plt.show()


### Reuse the K-Means pattern

These repeat the first complete K-Means example:

- `KMeans(...)` creates the model;
- `.fit_predict(digit_values)` learns and returns assignments;
- `.cluster_centers_` retrieves the learned centers;
- `np.bincount(...)` counts assignments.

### Your coding task

Reuse the methods demonstrated in the complete coordinate
example. Complete the cluster count, model creation,
`.fit_predict(...)`, learned-center retrieval, and cluster-size
calculation. Expected output:

```text
Cluster labels: (1797,)
Centers: (10, 64)
```


In [ ]:
digit_cluster_count = None  # TODO: use 10

digit_model = None  # TODO: create KMeans with:
# n_clusters=digit_cluster_count
# n_init=10
# max_iter=300
# random_state=42

digit_cluster_labels = None  # TODO: fit_predict digit_values
digit_cluster_centers = None  # TODO: use cluster_centers_
digit_cluster_sizes = None  # TODO: use np.bincount(...)

print("Cluster labels:", digit_cluster_labels.shape)
print("Centers:", digit_cluster_centers.shape)
print("Cluster sizes:", digit_cluster_sizes)


<details>
<summary>Hint 1</summary>

There are ten digit types. Create the model exactly as in the first
complete K-Means case, but use `digit_cluster_count`. After
`.fit_predict(...)`, read the learned centers and count the labels.
</details>

<details>
<summary>Hint 2: code shape</summary>

```python
model = KMeans(
    n_clusters=cluster_count,
    n_init=10,
    max_iter=300,
    random_state=42,
)
labels = model.fit_predict(input_rows)
centers = model.cluster_centers_
sizes = np.bincount(labels)
```

Replace the general names with the digit variables.
</details>


The provided cell reshapes every 64-number center into an
`8 × 8` image.

### Turn each center back into an image

- `digit_cluster_centers[cluster_number]` selects one learned
  64-number center;
- `.reshape(8, 8)` restores its image shape.

The remaining plotting lines are provided. Focus on the data
change from one 64-number center back to one visible image.


In [ ]:
figure, axes = plt.subplots(2, 5, figsize=(10, 4))
for cluster_number, axis in enumerate(axes.ravel()):
    center_image = digit_cluster_centers[
        cluster_number
    ].reshape(8, 8)
    axis.imshow(center_image, cmap="gray")
    axis.set_title(f"Cluster {cluster_number}")
    axis.axis("off")

figure.suptitle("Average image at each K-Means center")
plt.tight_layout()
plt.show()


**Think about the output:**

1. Which centers resemble recognizable digits?
2. Does cluster number `0` necessarily represent digit zero?
3. Why might one center look like a mixture of writing styles?


# Case 4 — Cluster Spoken-Digit Recordings

## From pictures to sound

In the previous case, every handwritten image became 64 pixel
numbers. A recording is also stored as numbers: the microphone
measures the sound wave many times each second.

Recordings have different lengths, but K-Means needs every row
to have the same number of columns. We therefore use **MFCC
features**. MFCC features are a short numerical summary of how
a recording sounds. We will use 13 numbers for every recording.
We use the provided function to calculate them; we do not need
to calculate the sound mathematics by hand.

## Dataset description

The
[Free Spoken Digit Dataset](https://github.com/Jakobovski/free-spoken-digit-dataset)
contains 3,000 real WAV recordings of the English words zero
through nine:

- 6 speakers;
- 10 spoken digits;
- 50 recordings of each digit from each speaker;
- an 8,000 Hz sample rate;
- filenames such as `7_jackson_12.wav`.

The filename means: spoken digit 7, speaker Jackson, recording
number 12. To keep the Colab activity quick, this lab uses the
first 10 repetitions from every speaker and digit: 600
recordings in total.



## Input

- 600 short spoken-digit recordings;
- each recording becomes 13 MFCC numbers;
- model-input shape: `(600, 13)`.

## Output

- one K-Means cluster number per recording: shape `(600,)`;
- ten sound centers: shape `(10, 13)`;
- a comparison between real spoken digits and cluster numbers;
- a cluster suggestion for a recording made in Colab.

## Prepare the audio tools and data

The provided setup code checks whether `librosa` is installed,
installs it only when necessary, downloads the dataset once,
and prepares the selected file paths. Students do not need to
memorize that environment and file-management code.

Focus on the audio pipeline:

- `librosa.load(...)` reads a sound file as waveform numbers.
- `librosa.feature.mfcc(...)` creates the 13-number sound
  summary. `n_mfcc=13` requests 13 features, and `n_fft=512`
  uses short sound windows that fit these brief recordings.
- `Audio(...)` creates a playback control in the notebook.

The provided visualization shows the waveform and MFCC values
for one recording before the model uses them.


In [ ]:
# These modules let Python check and, when necessary,
# install a package in the current notebook environment.
import importlib.util
import subprocess
import sys

# Install librosa only if this notebook cannot find it.
# sys.executable points pip to the current notebook's Python.
if importlib.util.find_spec("librosa") is None:
    print(
        "librosa is missing. Installing it in the "
        "current notebook environment..."
    )
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "librosa",
        ]
    )
    importlib.invalidate_caches()
else:
    print("librosa is already installed.")

# These helpers manage folders, downloads, and ZIP extraction.
from pathlib import Path
from tempfile import gettempdir
from urllib.request import urlretrieve
from zipfile import ZipFile

# librosa reads audio and creates sound features.
# Audio and display add a playback control to the notebook.
import librosa
import librosa.display
from IPython.display import Audio, display

# Colab stores temporary files in /content. Outside Colab,
# use a temporary practicalai_day3 folder.
if Path("/content").exists():
    audio_runtime_directory = Path("/content")
else:
    audio_runtime_directory = (
        Path(gettempdir()) / "practicalai_day3"
    )
audio_runtime_directory.mkdir(
    parents=True,
    exist_ok=True,
)

audio_dataset_directory = (
    audio_runtime_directory
    / "free-spoken-digit-dataset-master"
    / "recordings"
)
# Download and unzip the public dataset only when its
# extracted recordings folder is missing.
if not audio_dataset_directory.exists():
    audio_zip_path = (
        audio_runtime_directory / "spoken_digits.zip"
    )
    urlretrieve(
        "https://github.com/Jakobovski/"
        "free-spoken-digit-dataset/archive/"
        "refs/heads/master.zip",
        audio_zip_path,
    )
    with ZipFile(audio_zip_path, "r") as audio_zip:
        audio_zip.extractall(audio_runtime_directory)

# Find every WAV file, then keep a balanced 600-file subset.
all_audio_paths = sorted(
    audio_dataset_directory.glob("*.wav")
)
audio_paths = []
for audio_path in all_audio_paths:
    repetition = int(
        audio_path.stem.rsplit("_", 1)[1]
    )
    if repetition < 10:
        audio_paths.append(audio_path)

sample_audio_path = audio_paths[0]
# Convert one WAV file into waveform numbers at 8,000 Hz.
sample_waveform, sample_rate = librosa.load(
    sample_audio_path,
    sr=8000,
)
# Convert that waveform into 13 MFCC sound-feature rows.
sample_mfcc = librosa.feature.mfcc(
    y=sample_waveform,
    sr=sample_rate,
    n_mfcc=13,
    n_fft=512,
)

print("librosa version:", librosa.__version__)
print("Complete dataset files:", len(all_audio_paths))
print("Lab subset files:", len(audio_paths))
print("Sample file:", sample_audio_path.name)
print("Sample rate:", sample_rate, "values per second")
display(Audio(filename=str(sample_audio_path)))

figure, axes = plt.subplots(2, 1, figsize=(10, 6))
librosa.display.waveshow(
    sample_waveform,
    sr=sample_rate,
    ax=axes[0],
)
axes[0].set_title("Waveform: sound values over time")

librosa.display.specshow(
    sample_mfcc,
    x_axis="time",
    ax=axes[1],
)
axes[1].set_title("Thirteen MFCC sound features")
axes[1].set_ylabel("MFCC number")
plt.tight_layout()
plt.show()


## Create one equal-length row per recording

We need to perform the same steps for 600 files, so a function
prevents us from copying the same code 600 times.

`extract_audio_features(...)`:

1. reads one WAV file;
2. calculates 13 MFCC features across time;
3. uses `.mean(axis=1)` to make one average number for each
   feature;
4. returns one row containing exactly 13 numbers.

The loop also reads the real digit from each filename. These
digit answers are saved only so we can inspect the clusters
afterward. They are not included in the K-Means input.


In [ ]:
def extract_audio_features(audio_path):
    waveform, sample_rate = librosa.load(
        audio_path,
        sr=8000,
    )
    mfcc_values = librosa.feature.mfcc(
        y=waveform,
        sr=sample_rate,
        n_mfcc=13,
        n_fft=512,
    )
    return mfcc_values.mean(axis=1)


audio_feature_rows = []
audio_true_digits = []

for audio_path in audio_paths:
    audio_feature_rows.append(
        extract_audio_features(audio_path)
    )
    true_digit = int(audio_path.name.split("_")[0])
    audio_true_digits.append(true_digit)

audio_features = np.array(audio_feature_rows)
audio_true_digits = np.array(audio_true_digits)

print("Audio input shape:", audio_features.shape)
print("Saved answers shape:", audio_true_digits.shape)
print("First 13-number row:")
print(np.round(audio_features[0], 2))


## Put the sound features on comparable scales

Different MFCC columns can use different numerical ranges.
Without scaling, a column with larger numbers could have too
much influence on K-Means distance.

`StandardScaler()` creates a scaling tool.
`.fit_transform(...)` learns each training column's mean and
spread, then converts the columns to comparable scales.

### Scale and cluster the audio

- `.fit_transform(audio_features)` learns and applies one scale
  per MFCC column;
- `KMeans(...)`, `n_clusters`, `n_init`, `max_iter`, and
  `random_state` have the same meanings as in the complete
  coordinate example;
- `.fit_predict(audio_features_scaled)` learns sound centers and
  returns cluster numbers;
- `.cluster_centers_` retrieves the learned sound centers.

`np.bincount(...)` only summarizes the output and is already
familiar from the coordinate example.

### Your coding task

After the provided scaling lines, create a regular K-Means
model with ten clusters, run `.fit_predict(...)`, retrieve its
centers, and calculate cluster sizes. The real digit answers
must not be passed into the model.

Expected output:

```text
Audio cluster labels: (600,)
Audio centers: (10, 13)
```


In [ ]:
from sklearn.preprocessing import StandardScaler

audio_scaler = StandardScaler()
audio_features_scaled = audio_scaler.fit_transform(
    audio_features
)

audio_cluster_count = None  # TODO: use 10
audio_model = None  # TODO: create KMeans with:
# n_clusters=audio_cluster_count
# n_init=10
# max_iter=300
# random_state=42

audio_cluster_labels = None  # TODO: fit_predict scaled rows
audio_cluster_centers = None  # TODO: use cluster_centers_
audio_cluster_sizes = None  # TODO: use np.bincount(...)

print(
    "Audio cluster labels:",
    audio_cluster_labels.shape,
)
print(
    "Audio centers:",
    audio_cluster_centers.shape,
)
print("Audio cluster sizes:", audio_cluster_sizes)


<details>
<summary>Hint 1</summary>

Scaling is already complete. Reuse the full K-Means pattern from
the coordinate and handwritten-digit cases. The model input is
`audio_features_scaled`, not the filename answers.
</details>

<details>
<summary>Hint 2: code shape</summary>

```python
model = KMeans(
    n_clusters=cluster_count,
    n_init=10,
    max_iter=300,
    random_state=42,
)
labels = model.fit_predict(scaled_input)
centers = model.cluster_centers_
sizes = np.bincount(labels)
```

Replace the general names with the audio variables.
</details>


## Inspect what K-Means grouped

`np.zeros((10, 10))` creates an empty counting table. Rows
represent the real spoken digits and columns represent the
K-Means cluster numbers. The answers are used only now, after
training, to understand the output.

`np.argmax(...)` finds the row with the largest count in each
cluster column. This lets us describe a cluster with the digit
heard most often inside it. It does not change how K-Means was
trained.


In [ ]:
audio_cluster_table = np.zeros(
    (10, 10),
    dtype=int,
)
for true_digit, cluster_number in zip(
    audio_true_digits,
    audio_cluster_labels,
):
    audio_cluster_table[
        true_digit,
        cluster_number,
    ] += 1

audio_cluster_digit_names = []
for cluster_number in range(10):
    most_common_digit = int(
        np.argmax(
            audio_cluster_table[:, cluster_number]
        )
    )
    audio_cluster_digit_names.append(most_common_digit)

plt.figure(figsize=(9, 7))
plt.imshow(
    audio_cluster_table,
    cmap="Blues",
    aspect="auto",
)
plt.colorbar(label="Number of recordings")
plt.xticks(range(10))
plt.yticks(range(10))
plt.xlabel("K-Means cluster number")
plt.ylabel("Real spoken digit, revealed after fitting")
plt.title("What is inside each audio cluster?")
plt.show()

print(
    "Most common real digit in each cluster:",
    audio_cluster_digit_names,
)


## Record or upload your own voice in Colab

The next provided cell connects the browser microphone to
Python. JavaScript is required because the microphone belongs
to the browser, not directly to Python. You do not need to
write or memorize the JavaScript.

- Keep `recording_method = "microphone"` to record for two
  seconds.
- Change it to `"upload"` to upload an existing audio file.
- If neither method works, the next cell automatically uses a
  dataset recording so the activity can continue.

When asked, allow microphone access and say one English digit.
Record only yourself. The file remains in the temporary Colab
runtime unless you intentionally download, save, or share it.

The recording and file-conversion code is provided. Focus on
its output: one WAV file in the same format as the dataset.
That file can then enter the same feature and model pipeline.


In [ ]:
# b64decode changes the browser's Base64 text back into bytes.
from base64 import b64decode

# Javascript sends recording instructions to the Colab browser.
from IPython.display import Javascript

# Choose "microphone" or "upload" as the audio source.
recording_method = "microphone"
recorded_audio_path = None

# The microphone bridge below works in hosted Google Colab.
try:
    from google.colab import files, output
    running_in_colab = True
except ImportError:
    running_in_colab = False
    print(
        "Direct recording is available in hosted Colab. "
        "The dataset fallback will be used here."
    )

if running_in_colab and recording_method == "microphone":
    # This JavaScript asks the browser for microphone access,
    # records two seconds, and returns a Base64 data URL.
    microphone_code = Javascript(
        '''
        async function recordAudio(seconds) {
          // Ask the browser for a live microphone stream.
          const stream =
            await navigator.mediaDevices.getUserMedia(
              {audio: true}
            );
          // Collect small audio pieces while recording.
          const recorder = new MediaRecorder(stream);
          const chunks = [];
          recorder.ondataavailable =
            event => chunks.push(event.data);
          recorder.start();
          await new Promise(
            resolve => setTimeout(
              resolve, seconds * 1000
            )
          );
          const stopped = new Promise(
            resolve => recorder.onstop = resolve
          );
          recorder.stop();
          await stopped;
          stream.getTracks().forEach(
            track => track.stop()
          );
          const blob = new Blob(
            chunks,
            {type: recorder.mimeType}
          );
          // Return the recorded file as a Base64 data URL
          // that Python can receive through Colab.
          const reader = new FileReader();
          return await new Promise(resolve => {
            reader.onloadend =
              () => resolve(reader.result);
            reader.readAsDataURL(blob);
          });
        }
        '''
    )
    # Load the recording function into the browser.
    display(microphone_code)
    try:
        print("Recording for two seconds...")
        # Run JavaScript in the browser and receive its result.
        recorded_data = output.eval_js(
            "recordAudio(2)"
        )
        recorded_webm_path = (
            audio_runtime_directory
            / "my_spoken_digit.webm"
        )
        recorded_audio_path = (
            audio_runtime_directory
            / "my_spoken_digit.wav"
        )
        # Decode the Base64 part and save the browser's WebM
        # recording as a temporary file.
        recorded_webm_path.write_bytes(
            b64decode(recorded_data.split(",")[1])
        )
        # FFmpeg converts WebM into the same 8,000 Hz,
        # one-channel WAV format used by the dataset.
        subprocess.run(
            [
                "ffmpeg",
                "-y",
                "-loglevel",
                "error",
                "-i",
                str(recorded_webm_path),
                "-ar",
                "8000",
                "-ac",
                "1",
                str(recorded_audio_path),
            ],
            check=True,
        )
        print("Your recording:")
        display(
            Audio(
                filename=str(recorded_audio_path)
            )
        )
    except Exception as recording_error:
        recorded_audio_path = None
        print(
            "Microphone recording was not available:",
            recording_error,
        )

elif running_in_colab and recording_method == "upload":
    # This alternative opens Colab's local-file upload window.
    uploaded_files = files.upload()
    if uploaded_files:
        uploaded_name = next(iter(uploaded_files))
        recorded_audio_path = Path(uploaded_name)
        print("Uploaded audio:")
        display(
            Audio(filename=str(recorded_audio_path))
        )


## Place the new recording in a cluster

`.transform(...)` applies the same scaling learned from the
dataset. `.predict(...)` finds the nearest learned K-Means
center and returns its cluster number.

We then report the digit that appeared most often in that
cluster and play the closest dataset recording. This is a
**cluster-based suggestion**, not a reliable speech-recognition
system. K-Means may group voices by speaker, pitch, accent, or
recording style instead of only by the spoken digit.


In [ ]:
if recorded_audio_path is None:
    prediction_audio_path = audio_paths[0]
    print(
        "Using a dataset recording because no new "
        "recording was available."
    )
else:
    prediction_audio_path = recorded_audio_path

new_audio_features = extract_audio_features(
    prediction_audio_path
)
new_audio_features_scaled = audio_scaler.transform(
    [new_audio_features]
).astype(audio_features_scaled.dtype)
predicted_audio_cluster = int(
    audio_model.predict(
        new_audio_features_scaled
    )[0]
)
suggested_digit = audio_cluster_digit_names[
    predicted_audio_cluster
]

distances_to_new_cluster = np.linalg.norm(
    audio_features_scaled
    - audio_cluster_centers[
        predicted_audio_cluster
    ],
    axis=1,
)
representative_index = int(
    np.argmin(distances_to_new_cluster)
)
representative_audio_path = audio_paths[
    representative_index
]

print(
    "Closest cluster:",
    predicted_audio_cluster,
)
print(
    "Most common digit in that cluster:",
    suggested_digit,
)
print(
    "This is a cluster suggestion, "
    "not guaranteed recognition."
)
print("Audio being placed:")
display(Audio(filename=str(prediction_audio_path)))
print("Closest representative dataset recording:")
display(
    Audio(
        filename=str(representative_audio_path)
    )
)


**Think about the output:**

1. Did the closest cluster's most common digit match the sound?
2. Why is a cluster number not automatically a digit answer?
3. What else besides the spoken digit can make two voices
   sound similar or different?
4. Why did we use the real digit labels only after fitting?


# Case 5 — A Shape K-Means Does Not Fit Well

## Dataset description

`make_moons(...)` generates two interleaving half-circle shapes
inside Colab. This artificial dataset is designed to reveal a
limitation of center-based clustering.



## Input

- 600 two-number coordinate rows;
- two curved moon shapes;
- shape `(600, 2)`.

## Output

- 600 K-Means cluster numbers;
- two learned centers;
- a graph that reveals the model's geometric limitation.

`make_moons(...)` creates the two curved sets of coordinate
points. We use it because the curves make an important
limitation of K-Means visible.

### Generate and cluster the moon-shaped data

- `make_moons(...)` generates the curved coordinate input;
- `n_samples=600` requests 600 rows;
- `noise=0.08` adds a small amount of variation;
- `random_state=42` repeats the same generated rows;
- `KMeans(...)`, `.fit_predict(...)`, `.cluster_centers_`, and
  `np.bincount(...)` repeat the complete first-case workflow.

### Your coding task

Complete the cluster count, create the K-Means model, assign
cluster labels, retrieve the centers, and count cluster sizes.


In [ ]:
from sklearn.datasets import make_moons

moon_points, unused_moon_groups = make_moons(
    n_samples=600,
    noise=0.08,
    random_state=42,
)

moon_cluster_count = None  # TODO: use 2
moon_model = None  # TODO: create KMeans with:
# n_clusters=moon_cluster_count
# n_init=10
# random_state=42
moon_labels = None  # TODO: fit_predict moon_points
moon_centers = None  # TODO: use cluster_centers_
moon_cluster_sizes = None  # TODO: use np.bincount(...)

print("Moon input shape:", moon_points.shape)
print("Moon output shape:", moon_labels.shape)
print("Moon centers:", moon_centers.shape)
print("Moon cluster sizes:", moon_cluster_sizes)


<details>
<summary>Hint 1</summary>

Request two clusters. The remaining model steps repeat the complete
coordinate example, but the input variable is `moon_points`.
</details>

<details>
<summary>Hint 2: code shape</summary>

```python
model = KMeans(
    n_clusters=cluster_count,
    n_init=10,
    random_state=42,
)
labels = model.fit_predict(input_rows)
centers = model.cluster_centers_
sizes = np.bincount(labels)
```

Replace the general names with the moon variables.
</details>


### Compare the input with the cluster output

The comparison code is provided. The left graph receives only
`moon_points`; the right graph also receives `moon_labels` and
`moon_centers`. Focus on whether center-based assignments match
the two curved shapes.


In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(
    moon_points[:, 0],
    moon_points[:, 1],
    color="#78909C",
    s=20,
)
axes[0].set_title("Input shape")

axes[1].scatter(
    moon_points[:, 0],
    moon_points[:, 1],
    c=moon_labels,
    cmap="viridis",
    s=20,
)
axes[1].scatter(
    moon_centers[:, 0],
    moon_centers[:, 1],
    color="red",
    marker="X",
    s=220,
)
axes[1].set_title("K-Means distance-based groups")

for axis in axes:
    axis.set_xlabel("Feature 1")
    axis.set_ylabel("Feature 2")

plt.tight_layout()
plt.show()


**Main lesson:** Code can run correctly while the algorithm's
assumptions do not match the data shape. K-Means uses distances
to centers, so curved and interlocking groups are difficult.


# Recursive K-Means

The phrase “recursive K-Means” can describe several custom
procedures. In this lab it means **Bisecting K-Means**:

```text
start with all rows in one group
→ split one group into two with K-Means
→ choose another current group
→ split it into two
→ continue until the requested number of final groups exists
```

Each internal split uses `K=2`. The final `n_clusters` may be
larger than two.


# Recursive K-Means: Complete Hierarchical Example

## Dataset description

`make_blobs(...)` generates 800 artificial coordinate rows
around four specified locations. The four compact groups are
arranged into two broader sides so repeated two-way splits are
visible.



## Input

- 800 coordinate rows;
- four compact subgroups arranged into two broad sides;
- shape `(800, 2)`;
- no target answers.

## Output

- one label per row;
- requested final cluster counts of 2, 3, and 4;
- center arrays and broad-to-detailed visualizations.

`BisectingKMeans(...)` parameters tell scikit-learn how many
final groups we want and which current group to split next.
The function creates a recursive, two-way-splitting K-Means
model. We import it in the next cell and create the models
after looking at the input shape.

### Build the hierarchical coordinate data

- `BisectingKMeans` is imported when it is first needed;
- `make_blobs(...)` is the same generator used in the first
  complete case;
- `n_samples=800` requests 800 rows;
- `centers=[...]` supplies four coordinate locations;
- `cluster_std=[...]` controls the four group spreads.

This repeats the earlier data-generation pipeline; the new
model is introduced after the input is ready.


In [ ]:
from sklearn.cluster import BisectingKMeans

nested_points, unused_nested_groups = make_blobs(
    n_samples=800,
    centers=[(-7, -2), (-3, 2), (3, -2), (7, 2)],
    cluster_std=[0.75, 0.8, 0.8, 0.75],
    random_state=42,
)

print("Recursive-example input shape:", nested_points.shape)


### Fit the three recursive models

- `n_clusters`: requested final number of groups;
- `bisecting_strategy="largest_cluster"`: split the largest
  current group next;
- `random_state=42`: reproduce internal split choices.

`.fit_predict(nested_points)` has the same purpose as it did
for regular K-Means: fit the model and return one cluster number
per row. `np.bincount(...)` then counts those assignments.


In [ ]:
recursive_2_model = BisectingKMeans(
    n_clusters=2,
    bisecting_strategy="largest_cluster",
    random_state=42,
)
recursive_3_model = BisectingKMeans(
    n_clusters=3,
    bisecting_strategy="largest_cluster",
    random_state=42,
)
recursive_4_model = BisectingKMeans(
    n_clusters=4,
    bisecting_strategy="largest_cluster",
    random_state=42,
)

recursive_2_labels = recursive_2_model.fit_predict(
    nested_points
)
recursive_3_labels = recursive_3_model.fit_predict(
    nested_points
)
recursive_4_labels = recursive_4_model.fit_predict(
    nested_points
)

print("2 final clusters:", np.bincount(recursive_2_labels))
print("3 final clusters:", np.bincount(recursive_3_labels))
print("4 final clusters:", np.bincount(recursive_4_labels))


### Draw the broad-to-detailed groups

The graph code is provided. It uses each fitted model's labels
and centers to show the change from 2 to 3 to 4 final clusters.
Focus on the broad-to-detailed splitting, not the Matplotlib
syntax.


In [ ]:
recursive_models = [
    (recursive_2_model, recursive_2_labels, 2),
    (recursive_3_model, recursive_3_labels, 3),
    (recursive_4_model, recursive_4_labels, 4),
]

figure, axes = plt.subplots(1, 3, figsize=(16, 4))

for axis, (model, labels, final_count) in zip(
    axes,
    recursive_models,
):
    axis.scatter(
        nested_points[:, 0],
        nested_points[:, 1],
        c=labels,
        cmap="viridis",
        s=20,
        alpha=0.7,
    )
    axis.scatter(
        model.cluster_centers_[:, 0],
        model.cluster_centers_[:, 1],
        color="black",
        marker="X",
        s=220,
    )
    axis.set_title(f"{final_count} final clusters")
    axis.set_xlabel("Feature 1")
    axis.set_ylabel("Feature 2")

figure.suptitle(
    "Recursive K-Means: broad groups to detailed groups"
)
plt.tight_layout()
plt.show()


# Case 6 — Recursive K-Means for Image Colors

## Dataset description

The flower is a **real sample image bundled with
scikit-learn**. `load_sample_image("flower.jpg")` reads the
installed copy directly, so Colab does not download it from an
external website. The image becomes a large dataset when every
pixel is treated as one row.



## Input

- one `427 × 640` color image;
- each pixel becomes `[red, green, blue]`;
- all-pixel input shape: `(273280, 3)`;
- fitting sample shape: `(13664, 3)`.

## Output

- one color-cluster number for all 273,280 pixels;
- eight RGB centers: shape `(8, 3)`;
- an image rebuilt using only eight learned colors.

A color image is also a dataset made of numbers. The function
`load_sample_image("flower.jpg")` loads a sample image bundled
with scikit-learn, so this case does not need an upload or an
internet download.

### Load and display the sample image

`load_sample_image("flower.jpg")` returns the bundled image as
a NumPy array. The display lines are provided; the important
result is `flower_image`, whose last dimension contains red,
green, and blue.


In [ ]:
from sklearn.datasets import load_sample_image

flower_image = load_sample_image("flower.jpg")

print("Image shape:", flower_image.shape)
plt.figure(figsize=(7, 5))
plt.imshow(flower_image)
plt.title("Original image")
plt.axis("off")
plt.show()


`reshape(-1, 3)` turns the image into pixel rows. `-1` asks NumPy
to calculate the required number of rows. Dividing by `255.0`
scales each RGB value into the range zero through one.

### Convert the image into pixel rows

`[::20]` selects every twentieth row for faster fitting. The
important pipeline change is:

```text
image → all RGB pixel rows → smaller fitting sample
```


In [ ]:
all_pixels = flower_image.reshape(-1, 3).astype(float) / 255.0
sample_pixels = all_pixels[::20]

print("All-pixel input:", all_pixels.shape)
print("Fitting sample:", sample_pixels.shape)


### Learn eight representative colors

The complete recursive coordinate example already demonstrated
`BisectingKMeans(...)` and its parameters:

- `n_clusters=8` requests eight final color groups;
- `bisecting_strategy="largest_cluster"` chooses the largest
  current color group for the next split;
- `.fit(sample_pixels)` learns from the smaller fitting sample;
- `.predict(all_pixels)` assigns all 273,280 pixel rows;
- `.cluster_centers_` retrieves the eight learned RGB colors;
- `np.bincount(...)` counts all pixel assignments.

### Your coding task

Complete the recursive model, `.fit(...)`, `.predict(...)`, and
cluster-size calculation.

Expected output:

```text
Pixel labels: (273280,)
Color centers: (8, 3)
```


In [ ]:
color_cluster_count = None  # TODO: use 8

color_model = None  # TODO: create BisectingKMeans with:
# n_clusters=color_cluster_count
# bisecting_strategy="largest_cluster"
# random_state=42

# TODO: fit color_model with sample_pixels
pixel_cluster_labels = None  # TODO: predict all_pixels
pixel_cluster_sizes = None  # TODO: use np.bincount(...)

print("Pixel labels:", pixel_cluster_labels.shape)
print("Color centers:", color_model.cluster_centers_.shape)
print("Pixels per color cluster:", pixel_cluster_sizes)


<details>
<summary>Hint 1</summary>

Reuse the `BisectingKMeans(...)` pattern from the complete recursive
coordinate example. Fit only `sample_pixels`, then use the fitted
model to predict `all_pixels`.
</details>

<details>
<summary>Hint 2: code shape</summary>

```python
model = BisectingKMeans(
    n_clusters=cluster_count,
    bisecting_strategy="largest_cluster",
    random_state=42,
)
model.fit(fitting_rows)
labels = model.predict(all_rows)
sizes = np.bincount(labels)
```

Replace the general names with the color variables.
</details>


### Rebuild the image from learned colors

- `color_model.cluster_centers_[pixel_cluster_labels]` replaces
  every pixel row with its assigned center color;
- `.reshape(flower_image.shape)` restores the original image
  height, width, and three color channels;
- `np.clip(values, 0, 1)` keeps display values in range.

The provided plot places the original and reconstructed images
side by side.


In [ ]:
compressed_pixels = color_model.cluster_centers_[
    pixel_cluster_labels
]
compressed_image = compressed_pixels.reshape(
    flower_image.shape
)
compressed_image = np.clip(compressed_image, 0, 1)

figure, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(flower_image)
axes[0].set_title("Original colors")
axes[0].axis("off")

axes[1].imshow(compressed_image)
axes[1].set_title("Eight recursive K-Means colors")
axes[1].axis("off")

plt.tight_layout()
plt.show()


### Display the learned color palette

The provided bar chart displays the eight RGB values stored in
`color_model.cluster_centers_`. Bar height has no measurement
meaning; only the learned colors matter.


In [ ]:
palette_colors = np.clip(
    color_model.cluster_centers_,
    0,
    1,
)

plt.bar(
    range(color_cluster_count),
    [1] * color_cluster_count,
    color=palette_colors,
    edgecolor="black",
)
plt.yticks([])
plt.xlabel("Color-cluster number")
plt.title("Eight learned RGB centers")
plt.show()


# Review

Answer in complete sentences:

1. What information is missing in unsupervised learning that was
   present in supervised learning?
2. What are the two main outputs of K-Means?
3. What happens during one K-Means update round?
4. Why are cluster numbers arbitrary?
5. Why does inertia usually decrease when K increases?
6. Why must audio recordings become equal-length feature rows?
7. Why is an audio cluster number not automatically a digit?
8. What did the moon-shaped dataset reveal?
9. How does Recursive/Bisecting K-Means build final groups?
10. In the image case, what did one input row represent?


# Optional Extensions

- Change the coordinate model from `K=3` to `K=4`.
- Change the digit model from 10 to 8 or 12 clusters.
- Record two different digits and compare their audio clusters.
- Change the image palette from 8 to 4 or 16 colors.
- Change `bisecting_strategy` to `"biggest_inertia"` and compare.

Record the changed input, expected output shape, observed
output, and one conclusion.


## Reference Connection

The `practicalAI` reference repository lists K-Means as a topic
but does not contain a completed K-Means notebook. This lab
reuses its established NumPy-array, pandas-style data inspection,
model, and visualization workflow while adding clustering with
current scikit-learn APIs.
